## Initial Setup

In [ ]:
import pandas as pd
import requests
import json
import math
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import requests
import geopandas as gpd
import re
from shapely.geometry import Point, Polygon, box
from shapely.ops import unary_union
from shapely.geometry import Point
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait as wait
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
from selenium.webdriver.common.keys import Keys
import csv
from datetime import date
from tqdm import tqdm
import copy
import os
from scipy.spatial import cKDTree
from matplotlib_scalebar.scalebar import ScaleBar

References:  
Google Places API Nearby Search(new): https://developers.google.com/maps/documentation/places/web-service/nearby-search?hl=ja  
SKU types: https://developers.google.com/maps/documentation/javascript/reference/place?hl=ja
Place types: https://developers.google.com/maps/documentation/places/web-service/place-types?hl=ja&_gl=1*1njkdfh*_up*MQ..*_ga*NTM0Njc2OTg4LjE3NjA0OTQ1NjU.*_ga_SM8HXJ53K2*czE3NjA0OTQ1NjUkbzEkZzAkdDE3NjA0OTQ1NjUkajYwJGwwJGgw#table-a

In [ ]:
# --- API Configuration ---
# API Key
global API_KEY
API_KEY = 'YOUR_API_KEY'  # Replace with your actual key

# Request Types
global searchtype
searchtype = ["bakery", "bar", "cafe", "night_club", "restaurant"] # Replace with types of your interest

# Field Mask (Requested columns; replace with fields of your interest)
global callfields
callfields = "places.id,places.displayName,places.adrFormatAddress,places.location,places.businessStatus,places.primaryType,places.regularOpeningHours,places.priceLevel,places.priceRange,places.rating,places.userRatingCount,places.googleMapsUri"

# Define Request Limit
max_req = 5000

In [ ]:
# --- local Cartesian coordinate system (CRS) ---
Local_CRS = 6677 #Change to your local epsg
crs_str = f'epsg:{Local_CRS}'

## Function definitions

In [ ]:
# API Request Function
def get_places_info(searchpoints,search_distance=50000):
    url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': API_KEY,
        'X-Goog-FieldMask': callfields
    }
    # Ensure CRS is WGS84
    if not searchpoints.crs == 'epsg:4326':
        searchpoints = searchpoints.to_crs(epsg=4326)

    places_list = []
    for i in searchpoints.index:
        center = searchpoints[i]
        center_y = center.y
        center_x = center.x

        payload = {
            "includedTypes": searchtype,
            "maxResultCount": 20,
            "languageCode": 'ja',
            "rankPreference": "DISTANCE",
            "locationRestriction": {
                "circle": {
                    "center": {
                        "latitude": center_y,
                        "longitude": center_x
                    },
                    "radius": search_distance
                }
            }
        }

        response = requests.post(url, headers=headers, data=json.dumps(payload))
        data = response.json()
        
        if response.status_code == 200:
            data = response.json()
            # Check if the 'places' key exists in the response
            if 'places' in data:
                data['places'] = [dict(item, ID=i) for item in data['places']] #Adding 'ID' to each place in the list
                places_list.extend(data['places'])
            else:
                print(f"Warning: 'places' key not found in response for search id {i}. Response: {data}")
        else:
            print(f"Error: Request failed with status code {response.status_code} for grid id {i}. Response: {response.text}")
    return places_list

In [ ]:
# Data Processing Function
def placejson_to_df(places_list):
    """
    Converts API JSON response to a Pandas DataFrame with English column names.
    """
    places_df = pd.DataFrame(columns=['place_id', 'name', 'address', 'latitude', 'longitude', 
        'business_status', 'primary_type', 
        'hours_mon', 'hours_tue', 'hours_wed', 'hours_thu', 'hours_fri', 'hours_sat', 'hours_sun',
        'price_level', 'price_min', 'price_max', 'rating', 'user_rating_count', 
        'searchid', 'url'])
    
    for place in places_list:
        def get_hours(day_idx):
            if 'regularOpeningHours' in place:
                return place['regularOpeningHours']['weekdayDescriptions'][day_idx].split(': ')[1]
            return ""
        places_df = pd.concat([places_df, pd.DataFrame([{
            'place_id': place['id'],
            'name': place['displayName']['text'],
            'address': re.sub('^.*</span> <span class="region">', '', place['adrFormatAddress']).replace('</span><span class="street-address">', '').replace('</span>', ''),
            'latitude': place['location']['latitude'],
            'longitude': place['location']['longitude'],
            'business_status': place.get('businessStatus', ''),
            'primary_type': place.get('primaryType', ''),
            'hours_mon': get_hours(0),
            'hours_tue': get_hours(1),
            'hours_wed': get_hours(2),
            'hours_thu': get_hours(3),
            'hours_fri': get_hours(4),
            'hours_sat': get_hours(5),
            'hours_sun': get_hours(6),
            'price_level': place.get('priceLevel', ''),
            'price_min': place['priceRange']['startPrice']['units'] if 'priceRange' in place else "",
            'price_max': place['priceRange']['endPrice']['units'] if 'priceRange' in place else "",
            'rating': place.get('rating', ""),
            'user_rating_count': place.get('userRatingCount', 0),
            'searchid': place['ID'],
            'url': place['googleMapsUri']
        }])], ignore_index=True)
    return places_df

In [ ]:
# Distance Calculation
def response_distance(places_df, searchpoints):
    """
    Calculates the effective radius for each search point based on retrieved POIs.
    """
    
    places_gdf = gpd.GeoDataFrame(places_df, geometry=gpd.points_from_xy(places_df.longitude, places_df.latitude), crs='epsg:4326')
    places_gdf = places_gdf.to_crs(epsg=Local_CRS)
    searchpoints = gpd.GeoDataFrame(geometry=searchpoints)
    # CRS conversion for distance calculation
    if not searchpoints.crs == crs_str:
        searchpoints = searchpoints.to_crs(epsg=Local_CRS)
    
    result_df = pd.DataFrame(columns=['searchid', 'max_distance', 'item_count'])
    
    for p_id, p_row in searchpoints.iterrows():
        circle_center = p_row['geometry']
        # Filter places found by this specific search point
        places_in_circle = places_gdf[places_gdf['searchid'] == p_id]

        if not places_in_circle.empty:
            max_distance = 0
    
            for _, place_row in places_in_circle.iterrows():
                distance = circle_center.distance(place_row['geometry'])
                max_distance = max(max_distance, distance)
    
            result_df = pd.concat([result_df, pd.DataFrame({
                'searchid': [p_id],
                'max_distance': [max_distance],
                'item_count': [len(places_in_circle)]
            })], ignore_index=True)
        else:
            result_df = pd.concat([result_df, pd.DataFrame({
                'searchid': [p_id],
                'max_distance': [0],
                'item_count': [0]
            })], ignore_index=True)
    return result_df

In [ ]:
# Plotting retrieved area
def plot_obtained(dissolved_gdf,searchpoints,results_distance,merged_circles_geoseries=None):    
    # Create a GeoDataFrame to store the merged circles
    if merged_circles_geoseries is None:
        merged_circles_gdf = gpd.GeoDataFrame(geometry=[], crs=crs_str)
    else:
        merged_circles_gdf = gpd.GeoDataFrame(geometry=merged_circles_geoseries, crs=crs_str)

    if not searchpoints.crs == crs_str:
        searchpoints = searchpoints.to_crs(epsg=Local_CRS)
    searchpoints_gdf = gpd.GeoDataFrame(geometry=searchpoints, crs=crs_str)
    # Create buffers based on effective radius
    for p_index, p_row in searchpoints_gdf.iterrows():
        p_id = p_index
        p_polygon = p_row['geometry']               
        if results_distance[results_distance['searchid'] == p_id]['item_count'].values[0] == 0:
            max_distance = 50000      
        else:
          max_distance = results_distance[results_distance['searchid'] == p_id]['max_distance'].values[0]
    
        circle = p_polygon.buffer(max_distance)
        merged_circles_gdf = pd.concat([merged_circles_gdf, gpd.GeoDataFrame({'geometry': [circle]}, crs=crs_str)], ignore_index=True)
    
    # Merge all circles
    merged_circle = unary_union(merged_circles_gdf.geometry)
    merged_circle_geoseries = gpd.GeoSeries([merged_circle], crs=crs_str)
    
    # Plot merged circles
    fig, ax = plt.subplots(figsize=(10, 10))
    dissolved_gdf.plot(ax=ax, color='green', alpha=0.5)
    merged_circle_geoseries.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)
    plt.show()

    return merged_circle_geoseries

In [ ]:
# Taking buffers of polygons in unacquired areas until taking a -1m buffer becomes impossible
# If features separate during this process, perform the operation for each separated feature.
def buffer_until_impossible(dissolved_gdf, merged_circle_geoseries, buffer_amount=-1, min_area=1000):

    # Extract unacquired areas
    if merged_circle_geoseries is not None:
        difference_area = dissolved_gdf.unary_union.difference(merged_circle_geoseries.unary_union)
    else:
        difference_area = dissolved_gdf.unary_union
    
    # Break down difference_area into Polygons
    if isinstance(difference_area, Polygon):
        polygons = gpd.GeoDataFrame(geometry=[difference_area], crs=crs_str)
    else:
        polygons = gpd.GeoDataFrame(geometry=list(difference_area.geoms), crs=crs_str)

    # Transform difference_area to GeoSeries
    if isinstance(polygons, gpd.GeoDataFrame):
      polygons_geoseries = polygons.geometry
    elif isinstance(polygons, gpd.GeoSeries):
      polygons_geoseries = polygons
    else:
        raise TypeError("polygons must be a GeoSeries or GeoDataFrame")
        
    """
    Continue taking buffers of polygons in unacquired areas until taking a -1m buffer becomes impossible
    """
    final_polygons = []

    for polygon in polygons_geoseries:
        current_polygons = [polygon]

        while current_polygons:
            next_polygons = []
            for poly in current_polygons:
                buffered = poly.buffer(buffer_amount)
                if not buffered.is_empty:
                    if buffered.geom_type == 'Polygon':
                      if  buffered.area > min_area:
                        next_polygons.append(buffered)
                      elif buffered.area <= min_area:
                        final_polygons.append(buffered)
                    elif buffered.geom_type == 'MultiPolygon':
                        next_polygons.extend(
                            [p for p in buffered.geoms if p.area > min_area]
                        )
                        final_polygons.extend([p for p in buffered.geoms if p.area <= min_area])
            current_polygons = next_polygons

    return gpd.GeoSeries(final_polygons, crs=polygons.crs)

In [ ]:
# Response-type function to confirm continuation
def ask_continue():
    while True:
        response = input("Do you want to continue？ (y/n): ").strip().lower()
        if response in {"y", "yes"}:
            return True
        elif response in {"n", "no"}:
            return False
        else:
            print("Invalid input. Enter'y' or 'n'.")

## Retrieving place data

Retrieval of place information using a greedy approach

#### Preparation for retrieval and the first round

In [ ]:
# Load the gpkg for the target area and merge it into a single feature
gdf = gpd.read_file(r'PATH_TO_YOUR_SEARCH_AREA.gpkg')
dissolved_gdf = gdf.dissolve()
# dissolved_gdf = dissolved_gdf.to_crs(crs_str)

# Check the result
# print(dissolved_gdf.head())

total_area = dissolved_gdf.area.sum()
search_history = []

In [ ]:
# Buffering for search points
buffered_polygons = buffer_until_impossible(dissolved_gdf, None, min_area=total_area/10)

# Obtain the list of center points
next_search_point_gdf = gpd.GeoSeries(buffered_polygons.centroid, crs=crs_str)
requests = len(next_search_point)

In [ ]:
# Retrieve place information *Caution: This involves API calls
places_list = get_places_info(next_search_point_gdf)

In [ ]:
# Convert to data frame
places_df = placejson_to_df(places_list)
places_df

In [ ]:
# Calculate the actual retrieval radius per request
results_distance = response_distance(places_df,next_search_point_gdf)
results_distance.loc[results_distance['item_count'] == 0, 'max_distance'] = 50000     
        
# Calculate and draw the retrieved area
merged_circle_geoseries = plot_obtained(dissolved_gdf,next_search_point_gdf,results_distance)

In [ ]:
search_history.append({
                'request_id': requests,
                'x': next_search_point_gdf.x,
                'y': next_search_point_gdf.y,
                'radius': results_distance['max_distance']
            })

In [ ]:
# Buffering for next search points
buffered_polygons = buffer_until_impossible(dissolved_gdf, merged_circle_geoseries)

In [ ]:
# Plot the next search points
fig, ax = plt.subplots(figsize=(10, 10))
dissolved_gdf.plot(ax=ax, color='blue', alpha=0.5)
merged_circle_geoseries.plot(ax=ax, color='green', edgecolor='black', alpha=0.5)
buffered_polygons.centroid.plot(ax=ax, color='yellow', edgecolor='black', alpha=0.5)
plt.show()

In [ ]:
next_search_point_gdf = gpd.GeoSeries(buffered_polygons.centroid, crs=crs_str)
len(next_search_point_gdf)

In [ ]:
# Drop duplicates
places_df = places_df.drop_duplicates(subset=['place_id'])
len(places_df)

#### Retrieve all data

In [ ]:
next_search_point_gdf2 = next_search_point_gdf

In [ ]:
# Retrieve all items
while len(next_search_point_gdf2) > 0 & True:
    print(f"The API will be called for {len(next_search_point_gdf2)} times.")
    if not ask_continue():
        print("Shutting down the program...")
        break
    # Retrieve data via nearby search from the specified location
    requests = requests+len(next_search_point_gdf2)

    if requests > max_req:
        print("Error: Request count exceeded request limit. Forcing exit.")
        requests = requests-len(next_search_point_gdf2)
        break
    
    places_list = get_places_info(next_search_point_gdf2)
    # Convert to data frame
    places_df_new = placejson_to_df(places_list)
    print(f"{len(places_df_new)} places obtained.")
    # Calculate the actual retrieval radius per request
    results_distance2 = response_distance(places_df_new,next_search_point_gdf2)
    results_distance2.loc[results_distance2['item_count'] == 0, 'max_distance'] = 50000     
    # Calculate and draw the retrieved area
    merged_circle_geoseries = plot_obtained(dissolved_gdf,next_search_point_gdf2,results_distance2,merged_circle_geoseries)
    search_history.append({
                'request_id': requests,
                'x': next_search_point_gdf2.x,
                'y': next_search_point_gdf2.y,
                'radius': results_distance['max_distance']
            })
    # Drop duplicates and add newly retrieved to places_df
    places_df = pd.concat([places_df,places_df_new])
    places_df = places_df.drop_duplicates(subset=['place_id'])
    # Buffering for next search points
    buffered_polygons = buffer_until_impossible(dissolved_gdf, merged_circle_geoseries)
    next_search_point_gdf2 = gpd.GeoSeries(buffered_polygons.centroid, crs=crs_str)

# Data requests for miner unexplored areas
difference_area = dissolved_gdf.unary_union.difference(merged_circle_geoseries.unary_union)
if isinstance(difference_area, Polygon):
    polygons = gpd.GeoDataFrame(geometry=[difference_area], crs=crs_str)
else:
    polygons = gpd.GeoDataFrame(geometry=list(difference_area.geoms), crs=crs_str)
if isinstance(polygons, gpd.GeoDataFrame):
  polygons_geoseries = polygons.geometry
elif isinstance(polygons, gpd.GeoSeries):
  polygons_geoseries = polygons
next_search_point_gdf2 = gpd.GeoSeries(polygons_geoseries.centroid, crs=crs_str)

print(f"The API will be called for {len(next_search_point_gdf2)} times.")
if not ask_continue():
    print("Shutting down the program...")
    break
# Retrieve data via nearby search from the specified location
requests = requests+len(next_search_point_gdf2)

if requests > max_req:
    print("Error: Request count exceeded request limit. Forcing exit.")
    requests = requests-len(next_search_point_gdf2)
    break

places_list = get_places_info(next_search_point_gdf2)
# Convert to data frame
places_df_new = placejson_to_df(places_list)
print(f"{len(places_df_new)} places obtained.")
# Calculate the actual retrieval radius per request
results_distance2 = response_distance(places_df_new,next_search_point_gdf2)
results_distance2.loc[results_distance2['item_count'] == 0, 'max_distance'] = 50000     
# Calculate and draw the retrieved area
merged_circle_geoseries = plot_obtained(dissolved_gdf,next_search_point_gdf2,results_distance2,merged_circle_geoseries)
search_history.append({
            'request_id': requests,
            'x': next_search_point_gdf2.x,
            'y': next_search_point_gdf2.y,
            'radius': results_distance['max_distance']
        })
# Drop duplicates and add newly retrieved to places_df
places_df = pd.concat([places_df,places_df_new])
places_df = places_df.drop_duplicates(subset=['place_id'])

len(places_df)

#### Save data

In [ ]:
# When it is acceptable for the place information to include locations outside the specified range
places_df.to_csv('YOUR_FILE_NAME.csv')

In [ ]:
# When deleting place information outside the specified range
places_gdf = gpd.GeoDataFrame(places_df, 
                       geometry=gpd.points_from_xy(places_df['longitude'], places_df['latitude']),
                       crs="EPSG:4326")
places_gdf = places_gdf.to_crs(crs_str)
places_df_within = places_df[places_gdf.intersects(dissolved_gdf.unary_union)]
places_df_within.to_csv('YOUR_FILE_NAME.csv')

## Simulate and compare the methods

#### Setup

In [ ]:
target_polygon=dissolved_gdf.to_crs(crs_str)
target_polygon=target_polygon.unary_union
total_area = dissolved_gdf.area.sum()

In [ ]:
poi_df = places_gdf

In [ ]:
lat_col = 'latitude' if 'latitude' in poi_df.columns  else '緯度'
lon_col = 'longitude' if 'longitude' in poi_df.columns  else '経度'
    
poi_gdf = gpd.GeoDataFrame(
    poi_df, 
    geometry=gpd.points_from_xy(poi_df[lon_col], poi_df[lat_col]),
    crs="EPSG:4326"
).to_crs(crs_str)

In [ ]:
poi_in_area = poi_gdf[poi_gdf.intersects(target_polygon)]
total_true_pois = len(poi_in_area)
print(f"Actual total number of places within the target area: {total_true_pois}")

In [ ]:
coords = np.array([(geom.x, geom.y) for geom in poi_gdf.geometry])
tree = cKDTree(coords)

In [ ]:
def simulate_api_call(x, y, tree, max_radius=50000, max_results=20):
    """
    Simulates the behavior when calling the API from the specified coordinates.
    Return value: (List of POI indices obtained, Effective radius covered)
    """
    # Return up to max_results results in order of proximity within max_radius
    distances, indices = tree.query([x, y], k=max_results, distance_upper_bound=max_radius)
    
    # Extract only valid results (exclude those where distance becomes inf if not found)
    valid_mask = distances != float('inf')
    valid_indices = indices[valid_mask]
    valid_distances = distances[valid_mask]
    
    # Determination effective radius
    if len(valid_distances) < max_results:
        # If fewer than 20 items were found, everything within max_radius has been retrieved (100% coverage).
        effective_radius = max_radius
    else:
        # If exactly 20 items are found, it only covers up to the distance of the 20th item.
        effective_radius = valid_distances[-1]
        
    return valid_indices.tolist(), effective_radius

#### Function definitions

In [ ]:
# Method A: Fixed Grid Method
def run_grid_method(target_polygon, tree, grid_size=100):
    print(f"--- Running: Fixed Grid Method (Grid Size: {grid_size}m) ---")
    
    if hasattr(target_polygon, 'unary_union'):
        target_polygon = target_polygon.unary_union
        
    minx, miny, maxx, maxy = target_polygon.bounds
    
    x_coords = np.arange(minx, maxx, float(grid_size))
    y_coords = np.arange(miny, maxy, float(grid_size))
    
    requests = 0
    unique_pois = set()
    search_history = []
    
    for x in x_coords:
        for y in y_coords:
            point = Point(x, y)
            if target_polygon.contains(point):
                requests += 1
                
                # Receive effective_radius
                indices, effective_radius = simulate_api_call(x, y, tree, max_radius=grid_size) 
                unique_pois.update(indices)
                
                # Save effective_radius to history
                search_history.append({
                    'request_id': requests,
                    'x': x,
                    'y': y,
                    'radius': effective_radius 
                })
                
    return requests, len(unique_pois), search_history


# Method B: Adaptive Quadtree Method
def run_quadtree_method(target_polygon, tree, max_radius=50000, min_box_size=1):
    print(f"--- Running: Adaptive Quadtree Method ---")
    
    if hasattr(target_polygon, 'unary_union'):
        target_polygon = target_polygon.unary_union
        
    requests = [0]
    unique_pois = set()
    search_history = []
    
    def search_box(current_box):
        if not current_box.intersects(target_polygon):
            return
            
        center = current_box.centroid
        bounds = current_box.bounds
        
        # Radius required to fully cover the bounding box
        input_radius = np.sqrt((bounds[2] - bounds[0])**2 + (bounds[3] - bounds[1])**2) / 2
        
        requests[0] += 1
        
        # Receive effective_radius
        indices, effective_radius = simulate_api_call(center.x, center.y, tree, max_radius=input_radius)
        unique_pois.update(indices)
        
        # Save effective_radius to history
        search_history.append({
            'request_id': requests[0],
            'x': center.x,
            'y': center.y,
            'radius': effective_radius
        })
        
        # Subdivide if 20 items are reached and the box can still be divided
        if len(indices) == 20 and (bounds[2] - bounds[0]) > min_box_size:
            minx, miny, maxx, maxy = bounds
            midx, midy = (minx + maxx) / 2.0, (miny + maxy) / 2.0
            
            search_box(box(minx, miny, midx, midy))
            search_box(box(midx, miny, maxx, midy))
            search_box(box(minx, midy, midx, maxy))
            search_box(box(midx, midy, maxx, maxy))

    initial_box = box(*target_polygon.bounds)
    search_box(initial_box)
    
    return requests[0], len(unique_pois), search_history

#### Perform comparison

In [ ]:
grid_req50, grid_uniq50, grid_history50 = run_grid_method(target_polygon, tree, grid_size=50)
grid_req250, grid_uniq250, grid_history250 = run_grid_method(target_polygon, tree, grid_size=250)
quad_req, quad_uniq, quad_history = run_quadtree_method(target_polygon, tree)

In [ ]:
greedy_req = requests
greedy_uniq = total_true_pois
greedy_history = search_history

In [ ]:
print("\n" + "="*50)
print("[Simulation Results Comparison]")
print("="*50)
print(f"Total actual POIs in target area: {total_true_pois}")
print("-" * 50)
print(f"1-1. Fixed Grid Method (Grid 50m)")
print(f"   - API Calls:    {grid_req50:,}")
print(f"   - Retrieved POIs: {grid_uniq50:,} (Ratio: {grid_uniq50/total_true_pois*100:.1f}%)")
print("-" * 50)
print(f"1-2. Fixed Grid Method (Grid 250m)")
print(f"   - API Calls:    {grid_req250:,}")
print(f"   - Retrieved POIs: {grid_uniq250:,} (Ratio: {grid_uniq250/total_true_pois*100:.1f}%)")
print("-" * 50)
print(f"2. Adaptive Quadtree Method")
print(f"   - API Calls:    {quad_req:,}")
print(f"   - Retrieved POIs: {quad_uniq:,} (Ratio: {quad_uniq/total_true_pois*100:.1f}%)")
print("-" * 50)
print(f"3. Proposal Method (Greedy Method)")
print(f"   - API Calls: {greedy_req:,} ")
print(f"   - Retrieved POIs:  {greedy_uniq:,} 件 (Ratio: {100:.1f}%)")
print("="*50)

In [ ]:
# Helper function to process history into GeoDataFrames for plotting
def prepare_plot_data(history_list, crs):
    df = pd.DataFrame(history_list)
    points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['x'], df['y']), crs=crs)
    circles = points.copy()
    circles['geometry'] = circles.geometry.buffer(circles['radius'])
    return points, circles

# Prepare data for all three methods
grid_pts, grid_cir = prepare_plot_data(grid_history250, crs_str)
quad_pts, quad_cir = prepare_plot_data(quad_history, crs_str)
greedy_pts, greedy_cir = prepare_plot_data(greedy_history, crs_str)

# Create a 1x3 subplot figure
fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharex=True, sharey=True)
poly_gdf = gpd.GeoDataFrame(geometry=[target_polygon], crs=crs_str)

plot_settings = [
    (axes[0], grid_pts, grid_cir, f"(a) Fixed grid (250m)\nCalls: {grid_req250:,} | Rate: {grid_uniq250/total_true_pois*100:.1f}%"),
    (axes[1], quad_pts, quad_cir, f"(b) Adaptive quadtree\nCalls: {quad_req:,} | Rate: {quad_uniq/total_true_pois*100:.1f}%"),
    (axes[2], greedy_pts, greedy_cir, f"(c) Proposed greedy\nCalls: {greedy_req:,} | Rate {100:.1f}%")
]

for ax, pts, cir, title in plot_settings:
    # Plot the base target area
    poly_gdf.plot(ax=ax, color='lightgrey', edgecolor='black', alpha=0.5)
    
    # Plot the searched areas (Circles)
    cir.plot(ax=ax, facecolor='none', edgecolor='blue', alpha=0.4, linewidth=0.8)
    
    # Plot the search center points
    pts.plot(ax=ax, color='red', markersize=2, alpha=0.7)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Longitude (Projected)', fontsize=12)
    ax.set_ylabel('Latitude (Projected)', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    
    # Ensure circles look perfectly round, not stretched
    ax.set_aspect('equal')

    # Add Scale Bar
    scalebar = ScaleBar(1, units="m", length_fraction=0.25, location="lower right")
    ax.add_artist(scalebar)

plt.suptitle('', fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()

# Save the figure
plt.savefig('sampling_strategies_comparison.png', dpi=300, bbox_inches='tight')
plt.show()